In [1]:
import os
import nest_asyncio
from dotenv import load_dotenv

nest_asyncio.apply()
load_dotenv("../.env")

True

In [2]:
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.agent import ReActAgent

In [3]:
# 1. Configuration
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
# Define the LLM
llm = GoogleGenAI(model="models/gemini-2.0-flash", api_key=os.getenv("GEMINI_API_KEY"))
# Define the Embedding Model
embed_model = GoogleGenAIEmbedding(model_name="models/gemini-embedding-2-preview", api_key=os.getenv("GEMINI_API_KEY"))
# Set them globally
Settings.llm = llm
Settings.embed_model = embed_model

In [4]:
# 2. Setup the "Tool" (Using what we built in Notebook 2)
# For simplicity, we create a base engine here, but imagine this as the 'Elite' engine
documents = SimpleDirectoryReader("../data").load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()

2026-03-22 11:49:47,652 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"


In [5]:
# Define the Tools
tools = [
    QueryEngineTool(
        query_engine=query_engine,
        metadata=ToolMetadata(
            name="devguardian_search",
            description="Searches the technical documentation for DevGuardian. Use this for all technical details."
        ),
    )
]
print("Tools Defined")

Tools Defined


In [6]:
# THE AGENT TOOLS 
agent = ReActAgent(
    tools=tools,
    llm=llm,
    verbose=True
)
print("REACT AGENT IS READY")

REACT AGENT IS READY


In [7]:
# Test the agent's reasoning
response = await agent.run(user_msg="What are the main security tools available in DevGuardian?")

# Print the Final Result
print(f"\nFINAL AGENT RESPONSE\n")
print(response)

2026-03-22 11:49:47,872 - INFO - Running step init_run
2026-03-22 11:49:47,877 - INFO - Step init_run produced event <class 'llama_index.core.agent.workflow.workflow_events.AgentInput'>
2026-03-22 11:49:47,877 - INFO - Running step setup_agent
2026-03-22 11:49:47,879 - INFO - Step setup_agent produced event <class 'llama_index.core.agent.workflow.workflow_events.AgentSetup'>
2026-03-22 11:49:47,879 - INFO - Running step run_agent_step
2026-03-22 11:49:47,884 - INFO - AFC is enabled with max remote calls: 10.
2026-03-22 11:49:48,824 - INFO - Step run_agent_step produced event <class 'llama_index.core.agent.workflow.workflow_events.AgentOutput'>
2026-03-22 11:49:48,825 - INFO - Running step parse_agent_output
2026-03-22 11:49:48,828 - INFO - Running step call_tool
2026-03-22 11:49:48,833 - INFO - Step parse_agent_output produced event <class 'NoneType'>
2026-03-22 11:49:49,509 - INFO - AFC is enabled with max remote calls: 10.
2026-03-22 11:49:50,592 - INFO - Step call_tool produced even


FINAL AGENT RESPONSE

DevGuardian includes a security scanning module that detects potential credential leaks like API keys, tokens, and passwords. This helps prevent the accidental exposure of sensitive information.
